---
title: "Liminal geometry: diffusion time dependence"
engine: julia
---

Load the simulator, MRIBuilder, and the plotting backend:

In [ ]:
using Random
using MCMRSimulator
using MRIBuilder
using CairoMakie

Random.seed!(1)
set_theme!(Theme(fontsize=20))

## 1. Create reusable diffusion-time sequences

For the next experiment, keep the b-value and echo time fixed while varying the
diffusion time. We use the short-pulse approximation: the diffusion gradients
are instantaneous, point along x, and do not have a finite duration `δ`. This
lets us specify `Δ` and b-value directly, without scanner constraints. We also
create a b=0 reference sequence at the same echo time to normalize the measured
signal.

In [ ]:
diffusion_times = [5.0, 10.0, 20.0, 40.0, 60.0]  # ms
fixed_bval = 1.0
fixed_echo_time = 100.0  # ms
diffusion_orientation = [1.0, 0.0, 0.0]

diffusion_time_sequences = [
    MRIBuilder.DiffusionSpinEcho(
        bval=fixed_bval,
        TE=fixed_echo_time,
        Δ=diffusion_time,
        gradient=(type=:instant, orientation=diffusion_orientation,),
    )
    for diffusion_time in diffusion_times
]

b0_reference_sequence = MRIBuilder.DiffusionSpinEcho(
    bval=0.0,
    TE=fixed_echo_time,
    Δ=first(diffusion_times),
    gradient=(type=:instant, orientation=diffusion_orientation,),
)

full_sequences = [b0_reference_sequence, diffusion_time_sequences...]

MRIBuilder.plot_sequence(first(diffusion_time_sequences))

```
MRIBuilder.plot_sequence(last(diffusion_time_sequences))
```

All five diffusion-weighted sequences have the same b-value (`3`) and echo time
(`100 ms`); only the diffusion time changes, from `5` to `60 ms`. Because the
gradients are instantaneous, the chosen `Δ` values can be shorter than the
finite gradient pulses used in a scanner-constrained acquisition. If we later
change b-value, only the gradient impulse strength needs to change; `Δ` and
`TE` can remain fixed.

## 2. Diffusion time dependence inside a neuron

We will first simulate the diffusion inside a single neuron morphology.
The easiest way to do this in MCMRSimulator is to use a liminal geometry. 
Liminal geometry generally represents a statistical population of cells, 
but if we set the extracellular fraction to zero, the simulated volume is entirely intracellular. 
This lets us simulate diffusion inside a single neuron morphology without having to explicitly position it in a box.
Load the neuron morphology from its SWC file, then use it as the only cell
template in a liminal geometry.

In [ ]:
neuron = read_swc(joinpath(@__DIR__, "cell_morphologies", "neuron.swc"))
neuron_geometry = LiminalGeometry(
    [(1.0, neuron)],
    extracellular_fraction=0.0,
)

There is also a sample "astrocyte.swc" or "microglia.swc" morphology in the `cell_morphologies` folder that can be used instead of the neuron.

Run the b=0 reference and the diffusion-time sequence set together. Diffusivity
and relaxation are held fixed; because all sequences have the same echo time,
the b=0 signal provides a common reference for the signal attenuation.

In [ ]:
neuron_simulation = Simulation(
    full_sequences;
    geometry=neuron_geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
    timestep=1.,
)

neuron_signals = readout(
    neuron_simulation;
    target_snr=10,
    subset=Subset(inside=true),
)

Compare the per-spin transverse signal for each diffusion time with the b=0
reference. The changing attenuation reflects how the neuron’s branched
intracellular morphology restricts diffusion over different time scales.

In [ ]:
neuron_transverse = transverse.(neuron_signals; mean=true)
neuron_attenuation = neuron_transverse[2:end] ./ first(neuron_transverse)

neuron_figure = Figure(size=(700, 400))
neuron_axis = Axis(
    neuron_figure[1, 1];
    xlabel="Diffusion time (ms)",
    ylabel="Transverse signal attenuation (S/S₀)",
)
scatterlines!(neuron_axis, diffusion_times, neuron_attenuation, markersize=12)
neuron_figure

## 3. Mix neuron, astrocyte, and microglia morphologies

Now include all three SWC cell types in the liminal population. Load each
morphology separately, then pass it into the `LiminalGeometry` constructor with
its relative cell-number weight:

In [ ]:
neuron = read_swc(joinpath(@__DIR__, "cell_morphologies", "neuron.swc"))
astrocyte = read_swc(joinpath(@__DIR__, "cell_morphologies", "astrocyte.swc"))
microglia = read_swc(joinpath(@__DIR__, "cell_morphologies", "microglia.swc"))

mixed_geometry = LiminalGeometry(
    geometries=[
        (6.0, neuron),
        (3.0, astrocyte),
        (1.0, microglia),
    ],
    extracellular_fraction=0.2,
)

The number weights sum to `10`, but `LiminalGeometry` normalizes them
internally to `0.6`, `0.3`, and `0.1`. They describe relative cell numbers,
not the volume fraction occupied by each cell type. The separate
`extracellular_fraction=0.2` sets the extracellular volume to 20%, leaving 80%
intracellular. See the [MCMRSimulator liminal geometry documentation](https://pages.fmrib.ox.ac.uk/ndcn0236/mcmrsimulator.jl/dev/liminal_geometry/#liminal_geometry).

Use the same sequence set and biophysical parameters to measure the average
signal across all spins in the mixed morphology:

In [ ]:
mixed_simulation = Simulation(
    full_sequences;
    geometry=mixed_geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
    timestep=1.0,
)

mixed_signals = readout(
    mixed_simulation;
    target_snr=10,
    max_spins=1000,
)

The readout includes both intracellular and extracellular spins. We average
their combined transverse signal per spin, then normalize it to the b=0
reference.

In [ ]:
mixed_transverse = transverse.(mixed_signals; mean=true)
mixed_attenuation = mixed_transverse[2:end] ./ first(mixed_transverse)

mixed_figure = Figure(size=(700, 400))
mixed_axis = Axis(
    mixed_figure[1, 1];
    xlabel="Diffusion time (ms)",
    ylabel="Transverse signal attenuation (S/S₀)",
)
scatterlines!(mixed_axis, diffusion_times, mixed_attenuation, markersize=12)
mixed_figure

Feel free to change the relative contribution of the different cell morphologies to see the effect on the results.

In more realistic simulations, you probably want to use a wider range of cell morphologies to sample the many different 
cell shapes in the brain.
Cell morphologies can be downloaded from the [NeuroMorpho.Org](https://neuromorpho.org/) database, as used here
or from single cell generators like the [Virtual Brain Cell Generator](https://github.com/palombom/Virtual-Cell-Generator-v1.0) or Octopus from [@brammerlohOCTOPUSVersatileOpensource2026].

# 4. Adding permeability to the processes
In the simulation above there is no exchange between the intra- and extracellular compartments.
This means that we could have just calculated the intracellular and extracellular signals separately and then combined them with the appropriate weights.
The liminal geometry becomes more useful when we add permeability to the processes, which allows spins to move between the intra- and extracellular compartments.

For each SWC-derived `FiniteCylinders` morphology, the first three objects
represent the soma and the remaining objects represent processes. Permeability
is a per-object surface property, so we assign a vector with zero permeability
for the soma and permeability `0.1` for each process. 
The value `0.1` is related to the exchange rate between compartments as described in section 2.5 of [@cottaarMultimodalMonteCarlo2026]. We modify copies of the morphologies
so the impermeable baseline remains available for comparison.

In [ ]:
process_permeability = 1e-3
function with_permeable_processes(morphology)
    permeable_morphology = deepcopy(morphology)
    permeability = ones(length(permeable_morphology))
    permeability[1:3] .= 0.0
    permeability[4:end] .= process_permeability
    permeable_morphology.permeability = permeability
    permeable_morphology
end

permeable_neuron = with_permeable_processes(neuron)
permeable_astrocyte = with_permeable_processes(astrocyte)
permeable_microglia = with_permeable_processes(microglia)

permeable_mixed_geometry = LiminalGeometry(
    geometries=[
        (6.0, permeable_neuron),
        (3.0, permeable_astrocyte),
        (1.0, permeable_microglia),
    ],
    extracellular_fraction=0.2,
)

Run the same acquisition and compare the combined mean signal with the
impermeable mixed-geometry result above:

In [ ]:
permeable_simulation = Simulation(
    full_sequences;
    geometry=permeable_mixed_geometry,
    diffusivity=2.0,
    R1=1 / 1000,
    R2=1 / 80,
    verbose=false,
    timestep=1.0,
)

permeable_signals = readout(permeable_simulation; target_snr=10)
permeable_transverse = transverse.(permeable_signals; mean=true)
permeable_attenuation = permeable_transverse[2:end] ./ first(permeable_transverse)

permeability_figure = Figure(size=(700, 400))
permeability_axis = Axis(
    permeability_figure[1, 1];
    xlabel="Diffusion time (ms)",
    ylabel="Transverse signal attenuation (S/S₀)",
)
scatterlines!(permeability_axis, diffusion_times, mixed_attenuation, label="Impermeable", markersize=10)
scatterlines!(permeability_axis, diffusion_times, permeable_attenuation, label="Permeable processes", markersize=10)
axislegend(permeability_axis, position=:rb)
permeability_figure

Feel free to change the `process_permeability` value to see how it affects the results.
Note that if the permeability is too high, the diffusion will become close to free diffusion, which means that the simulator will have to generate a very large number of spins to produce a reliable signal. 
This will make the simulation very slow, so it is best to keep the permeability low for this workshop.